# Hyperion Economy — lernende Agenten in Juno

Diese Demo trainiert drei lokale Q-Learning-Händler. Jeder Agent sieht dieselbe Hyperion-Wirtschaft, entscheidet sich für **halten**, **kaufen** oder **verkaufen** und speichert seine Erfahrung als kleine JSON-Datei. Es werden keine Online-Dienste und kein großes RL-Framework benötigt.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from learning_agents import train_agents, evaluate_agents, load_agent_memory

MEMORY_FILE = Path('agent_memory.json')
EPISODES = 30
YEARS = 20
SEED = 7

## 1. Training starten

Für eine schnelle Präsentation reichen 10 Episoden. Für sichtbar stabileres Verhalten sind 30–50 Episoden sinnvoll.

In [ ]:
result = train_agents(
    episodes=EPISODES, years=YEARS, seed=SEED, memory_path=str(MEMORY_FILE)
)
history = pd.DataFrame(result['history'])
print(f'{len(result["agents"])} Agenten trainiert.')
print(f'Gedächtnis: {MEMORY_FILE.resolve()}')
display(history.groupby('agent').tail(1)[['agent', 'final_value', 'trades', 'q_states', 'epsilon']])

## 2. Lernen sichtbar machen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for agent, frame in history.groupby('agent'):
    axes[0].plot(frame['episode'], frame['total_reward'].rolling(5, min_periods=1).mean(), label=agent)
    axes[1].plot(frame['episode'], frame['q_states'], label=agent)
axes[0].axhline(0, color='black', linewidth=0.8)
axes[0].set_title('Gleitender Lernfortschritt')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Belohnung')
axes[1].set_title('Erlernte Markt-Zustände')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Q-Table-Zustände')
for axis in axes:
    axis.legend()
    axis.grid(alpha=0.25)
plt.tight_layout()
plt.show()

## 3. Unabhängige Bewertung

Die Bewertung nutzt neue Zufallsjahre und schaltet Exploration aus. Damit sehen wir, was die Agenten aus ihrem Gedächtnis abrufen, statt sie während der Messung weiterlernen zu lassen.

In [ ]:
evaluation = pd.DataFrame(evaluate_agents(result['agents'], episodes=5, years=YEARS, seed=SEED + 1000))
summary = (evaluation.groupby('agent', as_index=False)
           .agg(avg_final_value=('final_value', 'mean'),
                avg_reward=('total_reward', 'mean'),
                avg_trades=('trades', 'mean')))
display(summary.sort_values('avg_final_value', ascending=False).round(2))

In [ ]:
fig, axis = plt.subplots(figsize=(10, 4))
evaluation.boxplot(column='final_value', by='agent', ax=axis, grid=False)
axis.set_title('Endwert in neuen Marktverläufen')
axis.set_xlabel('Agentenrolle')
axis.set_ylabel('Portfolio-Endwert')
plt.suptitle('')
plt.tight_layout()
plt.show()

## 4. Gedächtnis prüfen und später fortsetzen

In [ ]:
restored = load_agent_memory(str(MEMORY_FILE), seed=SEED)
memory_view = pd.DataFrame({
"agent": [agent.name for agent in restored],
"q_states": [len(agent.q_table) for agent in restored],
"epsilon": [agent.epsilon for agent in restored],
"known_actions": [sum(len(row) for row in agent.q_table.values()) for agent in restored],
})
display(memory_view)
print('Die Datei kann beim nächsten Juno-Lauf wieder geladen werden.')

## 5. Optional: interaktiver Kurztest

Wenn `ipywidgets` in Juno verfügbar ist, kann die Trainingslänge direkt auf dem iPad verändert werden. Ohne Widgets bleibt das feste Training oben vollständig nutzbar.

In [ ]:
try:
    import ipywidgets as widgets
    def quick_training(episodes=10, years=12):
        quick = train_agents(episodes=episodes, years=years, seed=SEED)
        frame = pd.DataFrame(quick['history'])
        display(frame.groupby('agent').tail(1)[['agent', 'final_value', 'trades', 'q_states']])
    display(widgets.interact(
        quick_training,
        episodes=widgets.IntSlider(value=10, min=5, max=60, step=5),
        years=widgets.IntSlider(value=12, min=5, max=30, step=1),
    ))
except ImportError:
    print('ipywidgets ist optional. Die festen Notebook-Zellen funktionieren trotzdem.')

### Demo-Erzählung

- **Profit-Scout** sucht stärker nach Rendite und akzeptiert Schwankungen.
- **Reserve-Keeper** wird für Risiko und instabile Jahre stärker bestraft.
- **Hyperion-Speculator** handelt opportunistisch, ohne ganz so risikoaggressiv zu sein.

Die Agenten sind absichtlich transparent: Der Zustand besteht aus Preisband, Stabilität, Liquidität und Bestand; das Gedächtnis ist eine einfache Q-Tabelle. Das ist eine gute Demo-Basis für spätere Erweiterungen wie Nachrichten-Agenten, Multi-Agenten-Handel oder komplexere Belohnungen.